# Unidad 3 · Colab 3 de 3
## Persistencia NoSQL con MongoDB y pymongo

**Objetivos de este notebook**

- Entender cuándo conviene NoSQL (documentos JSON) frente a una base relacional.
- Usar `pymongo` para hacer CRUD sobre colecciones de documentos.
- Escribir consultas y una agregación básica, pensadas para analítica web (eventos, logs).
- Comparar el modelo relacional (Colab 1 y 2) con el modelo de documentos para el caso de e-commerce.

> **Nivel:** intermedio.

> ⚠️ **Sobre este notebook:** no vamos a levantar un servidor de MongoDB real. Usamos [mongomock](https://github.com/mongomock/mongomock), una librería que imita la API de `pymongo` en memoria — así todo el código corre en Colab sin instalar nada ni crear una cuenta. Al final te mostramos cómo conectarte a un MongoDB real (por ejemplo, MongoDB Atlas, que tiene un nivel gratuito).

---

## 1. ¿Cuándo conviene NoSQL?

| | Relacional (SQL) | Documentos (MongoDB) |
|---|---|---|
| Estructura | Fija, definida por el esquema (columnas) | Flexible, cada documento puede tener campos distintos |
| Relaciones | Fuertes, con JOINs e integridad referencial | Débiles o embebidas dentro del documento |
| Caso de uso típico | Transacciones (pedidos, pagos, inventario) | Eventos de analítica, logs, catálogos con atributos variables |
| Escritura | Optimizada para consistencia | Optimizada para volumen y velocidad de escritura |

Para nuestro e-commerce: los pedidos (con integridad y montos) encajan mejor en PostgreSQL; los eventos de analítica web (cada pageview, cada click, con forma variable) encajan mejor en MongoDB.

Documentación oficial: [Manual de MongoDB](https://www.mongodb.com/docs/manual/) · [pymongo](https://pymongo.readthedocs.io/en/stable/)

## 2. Conceptos: base de datos, colección, documento

- **Base de datos:** agrupa colecciones (equivalente aproximado a una base SQL).
- **Colección:** agrupa documentos (equivalente aproximado a una tabla, pero sin esquema fijo).
- **Documento:** un JSON (técnicamente BSON) con un `_id` único generado automáticamente.

```json
{
  '_id': '...',
  'tipo_evento': 'pageview',
  'pagina': '/productos/1',
  'usuario_id': 42,
  'timestamp': '2026-03-01T10:15:00'
}
```

In [1]:
!pip install -q mongomock pymongo

import mongomock

cliente_mongo = mongomock.MongoClient()
db = cliente_mongo['ecommerce_analytics']
eventos = db['eventos']

print('Conexion (simulada) lista:', db.name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 807.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 17.8 MB/s eta 0:00:00
Conexion (simulada) lista: ecommerce_analytics


## 3. `insert_one` / `insert_many`

```python
eventos.insert_one({'tipo_evento': 'pageview', 'pagina': '/productos/1', 'usuario_id': 42})
```

Documentación oficial: [Insertar documentos](https://www.mongodb.com/docs/manual/tutorial/insert-documents/)

In [2]:
from datetime import datetime, timedelta

base = datetime(2026, 3, 1, 9, 0, 0)

eventos.insert_many([
    {'tipo_evento': 'pageview', 'pagina': '/productos/1', 'usuario_id': 42, 'timestamp': base},
    {'tipo_evento': 'pageview', 'pagina': '/productos/2', 'usuario_id': 42, 'timestamp': base + timedelta(minutes=1)},
    {'tipo_evento': 'click', 'pagina': '/productos/1', 'usuario_id': 42, 'elemento': 'boton_comprar', 'timestamp': base + timedelta(minutes=2)},
    {'tipo_evento': 'pageview', 'pagina': '/productos/1', 'usuario_id': 7, 'timestamp': base + timedelta(minutes=5)},
    {'tipo_evento': 'pageview', 'pagina': '/productos/3', 'usuario_id': 7, 'timestamp': base + timedelta(minutes=6)},
])
print(eventos.count_documents({}), 'eventos insertados')

5 eventos insertados


## 4. `find`: consultar documentos

```python
eventos.find({'tipo_evento': 'pageview'})
eventos.find({'pagina': '/productos/1', 'tipo_evento': 'pageview'})
eventos.find({'usuario_id': {'$in': [42, 7]}})
```

Operadores comunes: `$gt`/`$lt` (mayor/menor que), `$in` (dentro de una lista), `$regex` (coincidencia de texto).

Documentación oficial: [Consultas](https://www.mongodb.com/docs/manual/tutorial/query-documents/)

In [3]:
for doc in eventos.find({'tipo_evento': 'pageview', 'pagina': '/productos/1'}):
    print(doc)

{'tipo_evento': 'pageview', 'pagina': '/productos/1', 'usuario_id': 42, 'timestamp': datetime.datetime(2026, 3, 1, 9, 0), '_id': ObjectId('6aa017a5a6e046531d118ede')}
{'tipo_evento': 'pageview', 'pagina': '/productos/1', 'usuario_id': 7, 'timestamp': datetime.datetime(2026, 3, 1, 9, 5), '_id': ObjectId('6aa017a5a6e046531d118ee1')}


### Ejercicio 1 — Filtrar eventos

Escribí una consulta que devuelva todos los eventos de tipo `click` del `usuario_id` `42`.

<details>
<summary>💡 Ver solución</summary>

```python
for doc in eventos.find({'tipo_evento': 'click', 'usuario_id': 42}):
    print(doc)
```

</details>

In [4]:
for doc in eventos.find({'tipo_evento': 'click', 'usuario_id': 42}):
    print(doc)

{'tipo_evento': 'click', 'pagina': '/productos/1', 'usuario_id': 42, 'elemento': 'boton_comprar', 'timestamp': datetime.datetime(2026, 3, 1, 9, 2), '_id': ObjectId('6aa017a5a6e046531d118ee0')}


## 5. `update_one` / `delete_one`

```python
eventos.update_one({'_id': algun_id}, {'$set': {'procesado': True}})
eventos.delete_one({'_id': algun_id})
```

A diferencia de SQL, `update` en MongoDB necesita un operador como `$set` para indicar qué cambiar dentro del documento.

### Ejercicio 2 — Marcar eventos como procesados

Escribí el código para marcar todos los eventos de tipo `pageview` con un nuevo campo `procesado: True`, usando `update_many` (la versión de `update_one` que afecta a todos los documentos que matchean).

<details>
<summary>💡 Ver solución</summary>

```python
resultado = eventos.update_many({'tipo_evento': 'pageview'}, {'$set': {'procesado': True}})
print(resultado.modified_count, 'documentos actualizados')
```

</details>

In [5]:
resultado = eventos.update_many({'tipo_evento': 'pageview'}, {'$set': {'procesado': True}})
print(resultado.modified_count, 'documentos actualizados')

4 documentos actualizados


## 6. Agregación: resumir datos

El aggregation pipeline de MongoDB encadena etapas (`$match`, `$group`, `$sort`, etc.), similar en espíritu a `WHERE` + `GROUP BY` + `ORDER BY` en SQL.

```python
pipeline = [
    {'$match': {'tipo_evento': 'pageview'}},
    {'$group': {'_id': '$pagina', 'vistas': {'$sum': 1}}},
    {'$sort': {'vistas': -1}},
]
list(eventos.aggregate(pipeline))
```

Documentación oficial: [Aggregation Pipeline](https://www.mongodb.com/docs/manual/core/aggregation-pipeline/)

In [6]:
pipeline = [
    {'$match': {'tipo_evento': 'pageview'}},
    {'$group': {'_id': '$pagina', 'vistas': {'$sum': 1}}},
    {'$sort': {'vistas': -1}},
]

for doc in eventos.aggregate(pipeline):
    print(doc)

{'vistas': 2, '_id': '/productos/1'}
{'vistas': 1, '_id': '/productos/2'}
{'vistas': 1, '_id': '/productos/3'}


### Ejercicio 3 — Vistas por usuario

Escribí un pipeline que agrupe por `usuario_id` y cuente cuántos eventos (de cualquier tipo) generó cada uno, ordenado de mayor a menor.

<details>
<summary>💡 Ver solución</summary>

```python
pipeline = [
    {'$group': {'_id': '$usuario_id', 'total_eventos': {'$sum': 1}}},
    {'$sort': {'total_eventos': -1}},
]
list(eventos.aggregate(pipeline))
```

</details>

In [7]:
pipeline = [
    {'$group': {'_id': '$usuario_id', 'total_eventos': {'$sum': 1}}},
    {'$sort': {'total_eventos': -1}},
]
list(eventos.aggregate(pipeline))

[{'total_eventos': 3, '_id': 42}, {'total_eventos': 2, '_id': 7}]

# 1. Instalar el emulador en Colab
!pip install -q mongomock

import mongomock

# 2. Cliente y base de datos simulados en memoria
cliente_mongo = mongomock.MongoClient()
db = cliente_mongo['ecommerce_analytics']

# 3. Prueba de inserción y lectura
coleccion = db['pedidos']
coleccion.insert_one({"cliente": "Ana Gómez", "total": 250.0})

doc = coleccion.find_one({"cliente": "Ana Gómez"})
print("Conexión simulada exitosa:")
print(doc)

In [9]:
# 1. Instalar el emulador en Colab
!pip install -q mongomock

import mongomock

# 2. Cliente y base de datos simulados en memoria
cliente_mongo = mongomock.MongoClient()
db = cliente_mongo['ecommerce_analytics']

# 3. Prueba de inserción y lectura
coleccion = db['pedidos']
coleccion.insert_one({"cliente": "Ana Gómez", "total": 250.0})

doc = coleccion.find_one({"cliente": "Ana Gómez"})
print("Conexión simulada exitosa:")
print(doc)

Conexión simulada exitosa:
{'cliente': 'Ana Gómez', 'total': 250.0, '_id': ObjectId('6aa0181da6e046531d118ee4')}


## 8. De vuelta al e-commerce: ¿SQL o NoSQL?

Un sistema de e-commerce real suele combinar ambos:

- **PostgreSQL (Colab 2):** pedidos, pagos, stock — datos que necesitan consistencia fuerte y relaciones claras.
- **MongoDB:** eventos de analítica (qué páginas visita cada usuario, qué clickea), catálogos con atributos muy variables entre categorías de producto (un notebook tiene RAM y procesador; una remera tiene talle y color), logs de la aplicación.

No es una elección excluyente: son herramientas para problemas distintos dentro del mismo sistema.

In [10]:
import sqlite3

# Conexión local a base relacional (mismo estándar SQL que PostgreSQL)
con = sqlite3.connect(":memory:")
cur = con.cursor()
cur.execute("PRAGMA foreign_keys = ON;")

# Tablas relacionales con esquema estricto
cur.execute("""
CREATE TABLE productos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    precio REAL NOT NULL,
    stock INTEGER NOT NULL CHECK(stock >= 0)
);
""")

cur.execute("""
CREATE TABLE pedidos (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER NOT NULL,
    cantidad INTEGER NOT NULL,
    total REAL NOT NULL,
    FOREIGN KEY (producto_id) REFERENCES productos (id)
);
""")

# Carga inicial
cur.execute("INSERT INTO productos (nombre, precio, stock) VALUES (?, ?, ?)", ("Notebook Lenovo", 1200.0, 5))
con.commit()

# Transacción atómica: Descuenta stock y asienta el pedido en un solo bloque
def procesar_compra(producto_id: int, cantidad: int):
    try:
        cur.execute("BEGIN TRANSACTION;")

        # 1. Obtener precio y chequear stock actual
        cur.execute("SELECT precio, stock FROM productos WHERE id = ?", (producto_id,))
        producto = cur.fetchone()
        if not producto or producto[1] < cantidad:
            raise ValueError("Stock insuficiente o producto inexistente.")

        precio_unitario, stock_actual = producto
        monto_total = precio_unitario * cantidad

        # 2. Descontar stock
        cur.execute("UPDATE productos SET stock = stock - ? WHERE id = ?", (cantidad, producto_id))

        # 3. Registrar el pedido
        cur.execute("INSERT INTO pedidos (producto_id, cantidad, total) VALUES (?, ?, ?)",
                    (producto_id, cantidad, monto_total))

        con.commit()
        print(f"✔ Compra SQL exitosa: {cantidad} unidad(es) procesada(s). Total: ${monto_total:.2f}")
    except Exception as e:
        con.rollback()
        print(f"✖ Transacción cancelada: {e}")

procesar_compra(producto_id=1, cantidad=2)

# Consultar el estado final consistente
cur.execute("SELECT p.nombre, p.stock, ped.id, ped.total FROM productos p JOIN pedidos ped ON p.id = ped.producto_id")
print("Registro en BD relacional:", cur.fetchall())
con.close()

✔ Compra SQL exitosa: 2 unidad(es) procesada(s). Total: $2400.00
Registro en BD relacional: [('Notebook Lenovo', 3, 1, 2400.0)]


In [11]:
!pip install -q mongomock

import mongomock
from datetime import datetime

# Instancia de base NoSQL en memoria (API nativa de pymongo)
client = mongomock.MongoClient()
db = client["ecommerce_analytics"]

coleccion_catalogo = db["catalogo"]
coleccion_eventos = db["eventos_clickstream"]

# 1. Catálogo flexible: Documentos con atributos completamente distintos
productos_polimorficos = [
    {
        "sku": "TECH-001",
        "categoria": "notebooks",
        "nombre": "Notebook Pro 14",
        "precio": 1400.0,
        "especificaciones": {
            "ram_gb": 16,
            "procesador": "M2",
            "pantalla_pulgadas": 14.2
        }
    },
    {
        "sku": "APPAREL-002",
        "categoria": "indumentaria",
        "nombre": "Remera Algodón Estampada",
        "precio": 25.0,
        "especificaciones": {
            "talle": "L",
            "color": "Negro",
            "material": "100% Algodón"
        }
    }
]
coleccion_catalogo.insert_many(productos_polimorficos)

# 2. Eventos de telemetría/analítica (Clickstream de usuarios)
evento_navegacion = {
    "usuario_id": "usr_9812",
    "tipo_evento": "page_view",
    "timestamp": datetime.utcnow().isoformat(),
    "metadata": {
        "url": "/productos/notebooks/TECH-001",
        "dispositivo": "mobile",
        "tiempo_en_pantalla_segundos": 45
    }
}
coleccion_eventos.insert_one(evento_navegacion)

# Consulta sobre atributos dinámicos
notebook_filtrada = coleccion_catalogo.find_one({"especificaciones.ram_gb": 16}, {"_id": 0})
evento_registrado = coleccion_eventos.find_one({"usuario_id": "usr_9812"}, {"_id": 0})

print("✔ Producto con atributos anidados en MongoDB:\n", notebook_filtrada)
print("\n✔ Evento de analítica registrado:\n", evento_registrado)

✔ Producto con atributos anidados en MongoDB:
 {'sku': 'TECH-001', 'categoria': 'notebooks', 'nombre': 'Notebook Pro 14', 'precio': 1400.0, 'especificaciones': {'ram_gb': 16, 'procesador': 'M2', 'pantalla_pulgadas': 14.2}}

✔ Evento de analítica registrado:
 {'usuario_id': 'usr_9812', 'tipo_evento': 'page_view', 'timestamp': '2026-09-08T14:15:11.046254', 'metadata': {'url': '/productos/notebooks/TECH-001', 'dispositivo': 'mobile', 'tiempo_en_pantalla_segundos': 45}}


/tmp/ipykernel_533/68291828.py:44: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


## Mini-proyecto final: pipeline de analítica web

1. Insertá al menos 20 eventos simulados (`pageview`, `click`, `add_to_cart`) con distintos `usuario_id` y `pagina`, a lo largo de varios días.
2. Escribí un pipeline de agregación que calcule la tasa de conversión aproximada por página: `add_to_cart` dividido `pageview` para cada `pagina`.
3. Encontrá el `usuario_id` con más eventos totales.

**Entregable:** el código de inserción + los dos pipelines de agregación con su resultado.

---

**Fin de la Unidad 3.** Recorriste el ciclo completo: modelar y consultar datos relacionales con SQL puro, hacerlo de forma más productiva con un ORM, y usar un modelo de documentos NoSQL donde el esquema fijo no es lo que necesitás.

In [13]:
from datetime import datetime, timedelta
import random
import mongomock

# =====================================================================
# 1. Configuración de conexión y base de datos
# =====================================================================
cliente = mongomock.MongoClient()
db = cliente["analitica_web"]
eventos_col = db["eventos"]
eventos_col.delete_many({})

# =====================================================================
# 2. Generación e inserción de eventos simulados
# =====================================================================
random.seed(42)

usuarios = [f"usr_{i}" for i in range(1, 7)]
paginas = ["/home", "/productos/notebook", "/productos/teclado", "/checkout"]
tipos_evento = ["pageview", "click", "add_to_cart"]
pesos_evento = [0.55, 0.30, 0.15]

fecha_base = datetime(2026, 9, 1, 10, 0, 0)
eventos_simulados = []

for i in range(30):
    desplazamiento_dias = random.randint(0, 4)
    desplazamiento_minutos = random.randint(10, 600)
    marca_tiempo = fecha_base + timedelta(days=desplazamiento_dias, minutes=desplazamiento_minutos)

    usr = "usr_1" if (i % 3 == 0) else random.choice(usuarios)

    evento = {
        "usuario_id": usr,
        "pagina": random.choice(paginas),
        "tipo_evento": random.choices(tipos_evento, weights=pesos_evento)[0],
        "fecha": marca_tiempo
    }
    eventos_simulados.append(evento)

resultado_insercion = eventos_col.insert_many(eventos_simulados)
print(f"✔ Se insertaron {len(resultado_insercion.inserted_ids)} eventos en la colección 'eventos'.\n")

# =====================================================================
# 3. Pipeline 1: Compatible con mongomock (sin operador $round)
# =====================================================================
pipeline_conversion = [
    {
        "$group": {
            "_id": "$pagina",
            "total_pageviews": {
                "$sum": {"$cond": [{"$eq": ["$tipo_evento", "pageview"]}, 1, 0]}
            },
            "total_add_to_cart": {
                "$sum": {"$cond": [{"$eq": ["$tipo_evento", "add_to_cart"]}, 1, 0]}
            },
            "total_clicks": {
                "$sum": {"$cond": [{"$eq": ["$tipo_evento", "click"]}, 1, 0]}
            }
        }
    },
    {
        "$project": {
            "_id": 0,
            "pagina": "$_id",
            "total_pageviews": 1,
            "total_add_to_cart": 1,
            "total_clicks": 1,
            "tasa_conversion": {
                "$cond": [
                    {"$gt": ["$total_pageviews", 0]},
                    {"$divide": ["$total_add_to_cart", "$total_pageviews"]},
                    0.0
                ]
            }
        }
    },
    {
        "$sort": {"tasa_conversion": -1}
    }
]

print("=" * 75)
print("PIPELINE 1: TASA DE CONVERSIÓN POR PÁGINA (add_to_cart / pageview)")
print("=" * 75)
metricas_paginas = list(eventos_col.aggregate(pipeline_conversion))

for metrica in metricas_paginas:
    conv_pct = metrica["tasa_conversion"] * 100
    print(
        f"Página: {metrica['pagina']:<22} | "
        f"Views: {metrica['total_pageviews']:<2} | "
        f"Add-to-Cart: {metrica['total_add_to_cart']:<2} | "
        f"Conversión: {conv_pct:>6.2f}%"
    )

# =====================================================================
# 4. Pipeline 2: Usuario con mayor cantidad total de eventos
# =====================================================================
pipeline_usuario_top = [
    {
        "$group": {
            "_id": "$usuario_id",
            "total_eventos": {"$sum": 1},
            "paginas_visitadas": {"$addToSet": "$pagina"}
        }
    },
    {
        "$sort": {"total_eventos": -1}
    },
    {
        "$limit": 1
    },
    {
        "$project": {
            "_id": 0,
            "usuario_id": "$_id",
            "total_eventos": 1,
            "paginas_distintas": {"$size": "$paginas_visitadas"}
        }
    }
]

print("\n" + "=" * 75)
print("PIPELINE 2: USUARIO CON MÁS EVENTOS TOTALES")
print("=" * 75)
usuario_top = list(eventos_col.aggregate(pipeline_usuario_top))[0]
print(
    f"Usuario Líder: {usuario_top['usuario_id']} | "
    f"Total Eventos: {usuario_top['total_eventos']} | "
    f"Páginas distintas visitadas: {usuario_top['paginas_distintas']}"
)
print("=" * 75)

✔ Se insertaron 30 eventos en la colección 'eventos'.

PIPELINE 1: TASA DE CONVERSIÓN POR PÁGINA (add_to_cart / pageview)
Página: /checkout              | Views: 5  | Add-to-Cart: 0  | Conversión:   0.00%
Página: /home                  | Views: 0  | Add-to-Cart: 3  | Conversión:   0.00%
Página: /productos/notebook    | Views: 3  | Add-to-Cart: 0  | Conversión:   0.00%
Página: /productos/teclado     | Views: 8  | Add-to-Cart: 0  | Conversión:   0.00%

PIPELINE 2: USUARIO CON MÁS EVENTOS TOTALES
Usuario Líder: usr_1 | Total Eventos: 12 | Páginas distintas visitadas: 3
